# Part 1: Customer Churn Prediction Neural Network Analysis
This notebook covers the exploration, preprocessing, modeling, and evaluation of a feed-forward neural network to predict customer churn.

## Task 1: Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('../customer_churn_nn.csv' if os.path.exists('../customer_churn_nn.csv') else 'customer_churn_nn.csv')
print(f'Shape of dataset: {df.shape}')
df.info()

In [ ]:
print('Missing Values:\n', df.isnull().sum())
print('\nTarget Distribution:\n', df['churn'].value_counts(normalize=True))

## Task 2: Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

# One-hot encode
X = pd.get_dummies(X, columns=['region', 'plan_type', 'contract_type', 'payment_method'], drop_first=True).astype(float)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'X_train shape: {X_train_scaled.shape}')

## Task 3 & 4: Neural Network Model Building, Training, and Evaluation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Compute class weights to handle heavy imbalance
neg, pos = np.bincount(y_train)
weight_for_0 = (1 / neg) * (len(y_train) / 2.0)
weight_for_1 = (1 / pos) * (len(y_train) / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}

base_model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(1, activation='sigmoid')
])

base_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
history = base_model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test), epochs=30, batch_size=32, class_weight=class_weight, verbose=1)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = (base_model.predict(X_test_scaled) > 0.5).astype(int)
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## Task 5: Hyperparameter Experimentation

In [ ]:
# Code to demonstrate the comparison table of experiments run previously
results_df = pd.read_csv('results/model_comparison_table.csv' if os.path.exists('results/model_comparison_table.csv') else 'part-1-neural-network-analysis/results/model_comparison_table.csv')
results_df

## Task 6: Final Reflection
Refer to the README.md for the complete reflection and analytical details.